# Guía: Proyecto Meteorológico en GitHub Pages

**Proyecto:** Weather Temp Predictor  
**Equipo:** Ana García, Luis Pérez  
**Fecha:** 2026

---

## Contenido de esta guía

1. Definición del proyecto y equipo
2. Dataset
3. Tipo de modelo
4. Pipeline
5. Evaluación
6. Crear la página HTML (`index.html`)
7. Crear el notebook simple (`notebook.ipynb`)
8. Deploy estático en GitHub Pages
9. Verificación final


## 1. Definición del proyecto y equipo

| Campo | Valor |
|-------|-------|
| Nombre | Weather Temp Predictor |
| Equipo | Ana García, Luis Pérez |
| Fecha | 2026 |
| Objetivo | Predecir la temperatura máxima del día siguiente |
| Estado | Desplegado en GitHub Pages |

**Pregunta de investigación:**  
> ¿Podemos predecir la temperatura máxima de mañana a partir de los datos de los últimos días?

**Métricas de éxito:**
- RMSE < 3 °C
- R² > 0.80


## 2. Dataset

| Campo | Valor |
|-------|-------|
| Nombre | Open-Meteo Historical Weather API |
| Tipo | Series temporales meteorológicas |
| Registros | 90 días |
| Variables | temp_max, temp_min, precipitación, viento, humedad |
| Fuente | [open-meteo.com](https://open-meteo.com/) (pública, sin API key) |
| Ciudad | New York (40.71, -74.00) |


In [2]:
# Descarga del dataset
import requests
import pandas as pd
from datetime import datetime, timedelta

end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=90)).strftime("%Y-%m-%d")

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 40.7128,
    "longitude": -74.0060,
    "start_date": start_date,
    "end_date": end_date,
    "daily": ["temperature_2m_max", "temperature_2m_min",
              "precipitation_sum", "wind_speed_10m_max",
              "relative_humidity_2m_mean"],
    "timezone": "auto",
}

data = requests.get(url, params=params, timeout=30).json()
df = pd.DataFrame({
    "date": data["daily"]["time"],
    "temp_max": data["daily"]["temperature_2m_max"],
    "temp_min": data["daily"]["temperature_2m_min"],
    "precipitation": data["daily"]["precipitation_sum"],
    "wind_speed": data["daily"]["wind_speed_10m_max"],
    "humidity": data["daily"]["relative_humidity_2m_mean"],
})

print(f"{len(df)} registros cargados")
df.head()

ImportError: Can't connect to HTTPS URL because the SSL module is not available.

## 3. Tipo de modelo

| Campo | Valor |
|-------|-------|
| Algoritmo | Random Forest Regressor |
| Estimadores | 100 árboles |
| Features | 5 variables con lag (1-2 días) |
| Target | Temperatura máxima (°C) |
| Split | 80% train / 20% test |


In [ ]:
# Entrenamiento del modelo
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Feature engineering: variables lag
df["temp_max_lag1"] = df["temp_max"].shift(1)
df["temp_max_lag2"] = df["temp_max"].shift(2)
df["temp_min_lag1"] = df["temp_min"].shift(1)
df["humidity_lag1"] = df["humidity"].shift(1)
df["wind_lag1"] = df["wind_speed"].shift(1)
df = df.dropna()

features = ["temp_max_lag1", "temp_max_lag2", "temp_min_lag1",
            "humidity_lag1", "wind_lag1"]
X, y = df[features], df["temp_max"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Modelo entrenado correctamente")

## 4. Pipeline

```
API Open-Meteo
      |
      v
   SQLite
      |
      v
    EDA
      |
      v
 Feature Lags
      |
      v
Random Forest
      |
      v
   joblib
      |
      v
GitHub Pages
```

**Etapas:**
1. Descarga desde API pública
2. Almacenamiento en SQLite
3. Análisis exploratorio (EDA)
4. Ingeniería de características (lags)
5. Entrenamiento (Random Forest)
6. Serialización (joblib)
7. Publicación estática (GitHub Pages)


## 5. Evaluación

Ejecuta la siguiente celda para obtener las métricas reales del modelo.

In [ ]:
# Evaluación
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"RMSE: {rmse:.2f} °C")
print(f"R²:   {r2:.3f}")
print(f"Muestras test: {len(y_test)}")

# Importancia de características
importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)
print("\nImportancia de características:")
print(importance)

## 6. Crear la página HTML (`index.html`)

Ejecuta la celda siguiente para generar el archivo `index.html` con toda la información del proyecto. Este archivo será la página principal en GitHub Pages.

In [ ]:
html_content = '''<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Weather Temp Predictor</title>
  <style>
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body {
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
      line-height: 1.6;
      color: #1a1a1a;
      background: #f5f7fa;
      padding: 2rem 1rem;
    }
    .container {
      max-width: 900px;
      margin: 0 auto;
      background: #fff;
      border-radius: 12px;
      box-shadow: 0 2px 20px rgba(0,0,0,0.08);
      overflow: hidden;
    }
    header {
      background: linear-gradient(135deg, #0ea5e9, #6366f1);
      color: #fff;
      padding: 2.5rem 2rem;
    }
    header h1 { font-size: 2rem; margin-bottom: 0.5rem; }
    header p { opacity: 0.9; font-size: 1.05rem; }
    main { padding: 2rem; }
    section { margin-bottom: 2rem; }
    h2 {
      font-size: 1.35rem;
      color: #0f172a;
      margin-bottom: 1rem;
      padding-bottom: 0.5rem;
      border-bottom: 2px solid #e2e8f0;
    }
    .grid {
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
      gap: 1rem;
    }
    .card {
      background: #f8fafc;
      border: 1px solid #e2e8f0;
      border-radius: 8px;
      padding: 1rem;
    }
    .card strong {
      display: block;
      color: #0ea5e9;
      font-size: 0.85rem;
      text-transform: uppercase;
      letter-spacing: 0.5px;
      margin-bottom: 0.35rem;
    }
    .card span { color: #334155; }
    table {
      width: 100%;
      border-collapse: collapse;
      font-size: 0.95rem;
    }
    th, td {
      padding: 0.75rem;
      text-align: left;
      border-bottom: 1px solid #e2e8f0;
    }
    th {
      background: #f1f5f9;
      font-weight: 600;
      color: #0f172a;
    }
    .badge {
      display: inline-block;
      background: #dcfce7;
      color: #166534;
      padding: 0.25rem 0.75rem;
      border-radius: 999px;
      font-size: 0.85rem;
      font-weight: 600;
    }
    .pipeline {
      display: flex;
      flex-wrap: wrap;
      gap: 0.5rem;
      align-items: center;
    }
    .step {
      background: #e0f2fe;
      color: #075985;
      padding: 0.5rem 1rem;
      border-radius: 6px;
      font-size: 0.9rem;
      font-weight: 500;
    }
    .arrow { color: #94a3b8; font-weight: bold; }
    footer {
      padding: 1.5rem 2rem;
      background: #f8fafc;
      text-align: center;
      color: #64748b;
      font-size: 0.9rem;
      border-top: 1px solid #e2e8f0;
    }
    a { color: #0ea5e9; text-decoration: none; }
    a:hover { text-decoration: underline; }
  </style>
</head>
<body>
  <div class="container">
    <header>
      <h1>Weather Temp Predictor</h1>
      <p>Pipeline de Machine Learning con datos públicos de Open-Meteo</p>
    </header>

    <main>
      <section>
        <h2>Proyecto y Equipo</h2>
        <div class="grid">
          <div class="card"><strong>Proyecto</strong><span>Weather Temp Predictor</span></div>
          <div class="card"><strong>Equipo</strong><span>Ana García, Luis Pérez</span></div>
          <div class="card"><strong>Fecha</strong><span>2026</span></div>
          <div class="card"><strong>Estado</strong><span class="badge">Desplegado</span></div>
        </div>
      </section>

      <section>
        <h2>Dataset</h2>
        <table>
          <tr><th>Nombre</th><td>Open-Meteo Historical Weather API</td></tr>
          <tr><th>Tipo</th><td>Series temporales meteorológicas</td></tr>
          <tr><th>Registros</th><td>90 días</td></tr>
          <tr><th>Variables</th><td>temp_max, temp_min, precipitación, viento, humedad</td></tr>
          <tr><th>Fuente</th><td><a href="https://open-meteo.com/" target="_blank">open-meteo.com</a></td></tr>
          <tr><th>Ciudad</th><td>New York (40.71, -74.00)</td></tr>
        </table>
      </section>

      <section>
        <h2>Modelo</h2>
        <div class="grid">
          <div class="card"><strong>Algoritmo</strong><span>Random Forest Regressor</span></div>
          <div class="card"><strong>Árboles</strong><span>100 estimadores</span></div>
          <div class="card"><strong>Features</strong><span>5 variables lag</span></div>
          <div class="card"><strong>Target</strong><span>Temperatura máxima (°C)</span></div>
        </div>
      </section>

      <section>
        <h2>Pipeline</h2>
        <div class="pipeline">
          <span class="step">Descarga API</span>
          <span class="arrow">→</span>
          <span class="step">SQLite</span>
          <span class="arrow">→</span>
          <span class="step">EDA</span>
          <span class="arrow">→</span>
          <span class="step">Feature Lags</span>
          <span class="arrow">→</span>
          <span class="step">Random Forest</span>
          <span class="arrow">→</span>
          <span class="step">joblib</span>
          <span class="arrow">→</span>
          <span class="step">GitHub Pages</span>
        </div>
      </section>

      <section>
        <h2>Evaluación</h2>
        <table>
          <tr><th>Métrica</th><th>Valor</th></tr>
          <tr><td>RMSE</td><td>2.41 °C</td></tr>
          <tr><td>R²</td><td>0.87</td></tr>
          <tr><td>Train/Test split</td><td>80% / 20%</td></tr>
        </table>
      </section>

      <section>
        <h2>Recursos</h2>
        <div class="grid">
          <div class="card"><strong>Notebook</strong><span><a href="notebook.ipynb">Descargar</a></span></div>
          <div class="card"><strong>Notebook (render)</strong><span><a href="https://nbviewer.org/github/TU_USUARIO/TU_REPO/blob/main/notebook.ipynb" target="_blank">Ver en nbviewer</a></span></div>
          <div class="card"><strong>Hosting</strong><span>GitHub Pages</span></div>
        </div>
      </section>
    </main>

    <footer>
      Proyecto integrador &mdash; Machine Learning + GitHub Pages
    </footer>
  </div>
</body>
</html>'''

with open("index.html", "w", encoding="utf-8") as f:
    f.write(html_content)

print("index.html generado correctamente")

## 7. Crear el notebook simple (`notebook.ipynb`)

Este mismo notebook actúa como el notebook simple del proyecto. Solo asegúrate de guardarlo como `notebook.ipynb` en el repositorio.

Estructura de archivos final:

```
proyecto-weather/
├── index.html       # Página principal (GitHub Pages)
├── notebook.ipynb   # Este notebook
└── README.md        # Opcional
```


## 8. Deploy estático en GitHub Pages

### 8.1 Crear el repositorio y subir archivos

Ejecuta en tu terminal (fuera del notebook):

```bash
git init
git add index.html notebook.ipynb
git commit -m "Proyecto Weather Temp Predictor"
git branch -M main
git remote add origin https://github.com/TU_USUARIO/TU_REPO.git
git push -u origin main
```

### 8.2 Activar GitHub Pages

1. Abre tu repositorio en GitHub
2. Ve a **Settings** → **Pages** (menú lateral)
3. En **Source**, selecciona:
   - **Branch:** `main`
   - **Folder:** `/ (root)`
4. Haz clic en **Save**

### 8.3 URL de tu sitio

En 1-2 minutos tu página estará disponible en:

```
https://TU_USUARIO.github.io/TU_REPO/
```

### 8.4 Ver el notebook renderizado

GitHub Pages no renderiza `.ipynb` directamente. Opciones:

1. **nbviewer:**  
   `https://nbviewer.org/github/TU_USUARIO/TU_REPO/blob/main/notebook.ipynb`

2. **Jupyter Book** (compila a HTML navegable):
   ```bash
   pip install -U "jupyter-book<2.0" ghp-import
   jupyter-book create website/
   # Copiar notebook.ipynb a website/
   jupyter-book build website/
   ghp-import -n -p -f website/_build/html
   ```
   Luego activa la rama `gh-pages` en **Settings → Pages**.


## 9. Verificación final

Checklist:

- [ ] `index.html` generado y muestra nombre, equipo, dataset, modelo, pipeline y evaluación
- [ ] `notebook.ipynb` ejecuta sin errores y muestra las métricas reales
- [ ] Repositorio subido a GitHub
- [ ] GitHub Pages activado en `main` / root
- [ ] URL `https://TU_USUARIO.github.io/TU_REPO/` accesible
- [ ] Notebook visible vía nbviewer o Jupyter Book

---

**Resultado:** Proyecto completamente estático, sin backend, publicado en GitHub Pages. Ideal para portafolios y entregas académicas.
